# comparing automatic and manual transcriptions of participant audio

In [1]:
import numpy as np
import pandas as pd
import quail
import pickle
import random
import re
import os
from num2words import num2words
from nltk.corpus import stopwords

### choose 5 participants at random to manually transcribe

In [2]:
# load in participants dict
with open('../../data/pickles/id_maps.p', 'rb') as file:
    id_maps = pickle.load(file)

# random5 = random.sample(id_maps.keys(), 5)
# print(random5)


# Random 5 chosen:
# ['MD-013119-A-01', 'MD-102218-A-07', 'MD-020119-A-04', 'MD-102218-B-01', 'MD-020119-B-03']

# MD-013119-A-01 did not complete experiment. Replacing with MD-013119-A-02
# MD-020119-B-03 did not complete experiment. Replacing with MD-020119-B-02

In [3]:
random5 = ['MD-013119-A-02', 'MD-102218-A-07', 'MD-020119-A-04', 'MD-102218-B-01', 'MD-020119-B-02']

# get ses1 & ses1 psiturk ids
rand5_turkids = {k: id_maps[k] for k in random5}

## manual transcriptions

### create directory structure

In [202]:
# man_dir = os.path.abspath('../../data/transcriptions/manual')
# for subid, runs in id_maps.items():
#     for ses, turkid in runs.items():
#         folder = os.path.join(man_dir, subid, turkid)
#         if not os.path.isdir(folder) and not os.path.isdir(os.path.join(man_dir,'drops',subid)):
#             os.makedirs(folder, exist_ok=True)

### load in manual transcripts

## automatic transcription

### create speech context from union of words in annotations

In [5]:
atlep1 = pd.read_pickle('../../data/annotations_dfs/atlep1.p')
atlep2 = pd.read_pickle('../../data/annotations_dfs/atlep2.p')
arrdev = pd.read_pickle('../../data/annotations_dfs/arrdev.p')

In [6]:
def meets_word_criteria(string):
    """
    Removes words with characters not wanted in auto transcriber speech context
    """
    
    good_word = True
    
    # remove words containing digits
    if any(char.isdigit() for char in string):
        good_word = False
    
    # remove words surrounded by single quotes and possessives (avoid duplicates in nested quotations & possessives)
    if string.startswith("'") or string.endswith("'") or string.endswith("'s"):
        good_word = False
        
    # remove unhelpful simple words
    if len(string) <= 2:
        good_word = False
    
    return good_word

In [7]:
def create_speech_context(df):
    """
    Creates episode-specific speech context from video annotations
    """
    
    # use Narrative details (internal and external), Characters on screen, Speech, Character speaking, and Setting
    word_cols = [df.columns[i] for i in [2,3,4,6,7,9]]
    
    # create single string of all text
    allwords = ' '.join(df.loc[:,word_cols].apply(lambda x: ' '.join(x.dropna()), axis=1).values.tolist())
    
    # remove all characters except spaces (catches \n and \t), letters, apostrophes, dashes
    no_punctuation = re.sub("[^\w\s'-]+", '', allwords)
    
    speech_context = []
    
    # split words into list
    for word in no_punctuation.split():
        # identify unique words that meet criteria
        if word.lower() not in speech_context and meets_word_criteria(word):
            speech_context.append(word.lower())

    # remove English stopwords
    speech_context_nostop = [word for word in speech_context if word not in stopwords.words('english')]
    
    return speech_context_nostop

In [8]:
atlep1_speech_context = create_speech_context(atlep1)
atlep2_speech_context = create_speech_context(atlep2)
arrdev_speech_context = create_speech_context(arrdev)

In [9]:
audiodir = os.path.abspath('../../data/audio/')
keypath = os.path.abspath('../../../google-credentials/cloud-speech-credentials.json')

# for sid, data in rand5_turkids.items():
#     print(sid + ':')
#     for ses, turkid in data.items():
        
#         # get corrrect audio file names and speech context by session and condition
#         if ses == 'session 1':
#             audiofiles = [turkid+'-recall.wav', turkid+'-prediction.wav']
#         elif ses == 'session 2':
#             audiofiles = [turkid+'-delayed.wav', turkid+'-recall.wav']

#         for audiofile in audiofiles:
            
#             # SKIP ALREADY FINISHED ONE
#             if audiofile != 'debughKp9W:debug1QFCO-recall.wav':
                
#                 # set correct episode speech context
#                 if session == 'session 2' and audiofile.endswith('-recall.txt'):
#                     if 'A' in sid:
#                         episode_context = atlep2_speech_context
#                     elif 'B' in sid:
#                         episode_context = arrdev_speech_context
#                     else:
#                         episode_context = atlep1_speech_context
            
#                 # find file in correct testroom dir
#                 path = os.path.join(audiodir,'room1',turkid, audiofile)
#                 if not os.path.isfile(path):
#                     path = os.path.join(audiodir,'room2',turkid, audiofile)
#                     if not os.path.isfile(path):
#                         print('AUDIO FILE NOT FOUND: ' + path)
#                         break

#                 # and decode the audio
#                 print('\tdecoding ' + ses + ': ' + audiofile + ' ...')
#                 quail.decode_speech(path=path, keypath=keypath, save=True, speech_context=episode_context,
#                                     enable_word_time_offsets=False)


In [10]:
with open('../../data/audio/room1/debughKp9W:debug1QFCO/debughKp9W:debug1QFCO-recall.wav.p', 'rb') as f:
    test = pickle.load(f)

### parse google cloud objects into text files

In [ ]:
def parse_response_object(filepath):
    """
    parses Google Cloud RecognizeResponse objects and returns string of full transcript
    """
    
    # load pickled response object
    with open(filepath, 'rb') as f:
        resp_obj = pickle.load(f)
    
    # list for full participant transcript
    full_transcript = []
    # loop over decoded chunks
    for chunk in resp_obj:
        # access transcript
        for result in chunk.results:
            single_transcript = result.alternatives[0].transcript
            # split to list
            transcript_aslist = single_transcript.split(' ')

            for i, word in enumerate(transcript_aslist):
                # convert digits to words
                if word.isdigit():
                    transcript_aslist[i] = num2words(int(word))
                # convert times to words
                if ':' in word:
                    transcript_aslist[i] = ' '.join([num2words(int(i)) for i in transcript_aslist[i].split(':')])
                # convert currency to words
                if '$' in word:
                    transcript_aslist[i] = num2words(word.strip('$').replace(',',''), to='currency', 
                                                     currency='USD').split(',')[0]

            full_transcript.append(' '.join(transcript_aslist))
            
    
    return ' '.join(full_transcript)

In [30]:
for sid, data in rand5_turkids.items():
    for ses, tid in data.items():
        
        # find file in correct test room
        path = os.path.join(audiodir,'room1',tid)
        if not os.path.isdir(path):
            path = os.path.join(audiodir,'room2',tid)
            if not os.path.isdir(path):
                print('ID not found: ' + tid)
                break
        
        # assign correct audio file types
        if os.path.isfile(os.path.join(path,tid+'-prediction.wav.p')):
            files = ['-prediction.wav.p', '-recall.wav.p']
        else:
            files = ['-delayed.wav.p', '-recall.wav.p']
        
        # parse each response object and save out text file
        for file in files:
            filepath = os.path.join(path,tid+file)
            transcript = parse_response_object(filepath)
            
            with open(filepath.rstrip('.p')+'.txt', 'w') as f:
                f.write(transcript)
            

### load in automatic transcriptions

In [ ]:
for sid, data in rand5_turkids.items():
    for ses, tid in data.items():
        if 